# Paso 2: ajuste a la traducción chileno → español neutro (Gemma 4 E4B)

Un solo notebook para todas las condiciones del experimento. Cambia `CONDICION` (o pásala con `lanzar.py -p CONDICION ...`, que corre cada condición en un kernel nuevo). En modo interactivo, **reinicia el entorno entre condiciones** para liberar la GPU:

| `CONDICION` | Modelo de partida | ¿Se entrena el paso 2? |
|---|---|---|
| `solo_paso2` | Gemma 4 E4B base | sí |
| `paso1+2` | base + adaptador del paso 1 fusionado | sí |
| `sin_ajuste` | Gemma 4 E4B base | no (línea base, cero ejemplos) |
| `solo_paso1` | base + adaptador del paso 1 fusionado | no |
| `it_solo_paso2` | Gemma 4 E4B **instruct** (`-it`) | sí |
| `it_sin_ajuste` | Gemma 4 E4B **instruct** (`-it`) | no (línea base, cero ejemplos) |

Las dos condiciones principales usan **exactamente los mismos datos, hiperparámetros y semilla**; solo cambia el modelo de partida. Así, la diferencia entre `paso1+2` y `solo_paso2` mide el aporte del paso 1.

**Datos**: `data/entrenamiento/dataset_chilenismos_opus.csv` (3.980 pares, 796 expresiones), dividido con el **split oficial del grupo** (`data/entrenamiento/splits/split_oficial.csv`): estratificado por expresión, de cada una 4 pares van a entrenamiento y 1 a validación o prueba (3.184 / 399 / 397). Toda expresión se ve en entrenamiento; validación y prueba miden traducir frases nuevas con expresiones conocidas.

**Formato**: en el modelo base (sin plantilla de chat) se usa un prompt fijo y la pérdida se calcula **solo sobre la respuesta**:

```
<bos>Traduce la frase del español de Chile al español neutro.
Chileno: {frase chilena}
Neutro: {frase neutra}<eos>
```

En las condiciones `it_*` se usa la **plantilla de chat** del modelo instruct, porque fue entrenado con ese formato (la pérdida también va solo sobre la respuesta, que cierra con el fin de turno `<turn|>`):

```
<bos><|turn>user
Traduce la siguiente frase del español de Chile al español neutro. Responde solo con la traducción.

{frase chilena}<turn|>
<|turn>model
{frase neutra}<turn|>
```

`it_solo_paso2` frente a `solo_paso2` mide si conviene partir del modelo instruct en vez del base. El paso 1 no se aplica al instruct: el adaptador del paso 1 se entrenó sobre los pesos del base.

**LoRA del paso 2**: por defecto en atención + FFN, porque traducir exige alinear y copiar desde la frase de entrada (trabajo de la atención). `MODULOS_PASO2 = ("mlp",)` sirve como ablación.

In [ ]:
%pip install -q -U "transformers>=5.5" "peft>=0.17" accelerate datasets huggingface_hub sacrebleu
%pip uninstall -q -y torchao   # el torchao de Colab (0.10) rompe a peft al crear el LoRA; no se usa

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | {p.total_memory / 1e9:.0f} GB | bf16: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Configuración
REPO_HF = "mariopedemonte12/chineutro-gemma4"   # repo privado con datos, checkpoints y resultados; None = solo local
DIR_TRABAJO = None   # None = salidas/gemma4 dentro del repo git, o /content/chineutro en Colab
CONDICION = "solo_paso2"   # "solo_paso2" | "paso1+2" | "sin_ajuste" | "solo_paso1" | "it_solo_paso2" | "it_sin_ajuste"
MODELO_BASE = "google/gemma-4-E4B"
MODELO_IT = "google/gemma-4-E4B-it"
NOMBRE_PASO1 = "mlp_r64_5M_priotw80"         # carpeta de DIR_TRABAJO/paso1 con el adaptador a fusionar

MODULOS_PASO2 = ("atencion", "mlp")
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LR = 2e-4
EPOCAS = 3
LOTE = 16
MAX_LEN = 160
SEED = 42
DTYPE = "bfloat16"                   # "float32" solo para pruebas en GPUs sin bf16 (T4)
MAX_PASOS = -1                       # > 0 corta el entrenamiento en ese paso (pruebas)

### Datos, checkpoints y resultados: repo privado del Hugging Face Hub

Todo vive en el repo privado `REPO_HF`, que es un espejo de `DIR_TRABAJO`: `DIR_TRABAJO/x/y` queda en `x/y` del repo.

| Ruta en el repo | Contenido |
|---|---|
| `datos/` | `comun.py`, `lanzar.py`, los notebooks, `rag_documentos.jsonl`, `dataset_chilenismos_opus.csv`, `split_oficial.csv` |
| `indice_corpus.parquet`, `config_paso1.json`, `muestra_*.parquet`, `presencia_diccionario_*.csv` | salidas de la exploración (00) |
| `paso1/corpus_<M>M.parquet` | corpus del paso 1 ya construido |
| `paso1/<nombre>/last-checkpoint/` | último checkpoint del paso 1 (se reemplaza en cada guardado) |
| `paso1/<nombre>/adaptador/`, `metricas_paso1.json` | adaptador final y métricas |
| `paso2/<condición>/` | adaptador, predicciones y métricas de cada condición |
| `ejecuciones/` | notebook ejecutado y log de cada corrida sin interfaz (`lanzar.py`) |

**Sin interfaz** (runtime de Colab por API): `python datos/lanzar.py <notebook> --repo REPO_HF -p CONDICION paso1+2`. Corre el notebook con papermill y sube el log cada 10 minutos.

**Dentro del repo git** (local) se usan los datos del repo y `DIR_TRABAJO = salidas/gemma4`. Con `REPO_HF = None` no se sube ni se baja nada.

El token de Hugging Face (variable `HF_TOKEN`, `hf auth login` o el secreto `HF_TOKEN` de Colab) necesita **permiso de escritura** para subir resultados, y acceso al corpus restringido.

In [ ]:
# Entorno: token de Hugging Face, comun.py y los datos, y el repo del Hub
import os, sys, json
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

# Sin interfaz no se puede pedir el token: tiene que estar en HF_TOKEN, en el archivo de `hf auth login`
# o en los secretos de Colab
from huggingface_hub import get_token, snapshot_download
if not get_token() and EN_COLAB:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert get_token(), "Falta el token de Hugging Face (HF_TOKEN o hf auth login)"


def raiz_repo():
    """Raíz del repo git, si el notebook corre dentro de él."""
    actual = Path.cwd().resolve()
    return next((c for c in [actual, *actual.parents] if (c / "models/gemma4/comun.py").is_file()), None)


RAIZ_REPO = raiz_repo()
if DIR_TRABAJO is None:
    DIR_TRABAJO = RAIZ_REPO / "salidas/gemma4" if RAIZ_REPO else "/content/chineutro"
DIR_TRABAJO = Path(DIR_TRABAJO)
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)

if RAIZ_REPO:   # dentro del repo git: datos y código del repo
    RAIZ, DIR_CODIGO = RAIZ_REPO, RAIZ_REPO / "models/gemma4"
else:           # Colab: comun.py y los datos vienen de datos/ en el repo del Hub
    if REPO_HF:
        snapshot_download(REPO_HF, allow_patterns=["datos/*"], local_dir=DIR_TRABAJO)
    RAIZ = DIR_CODIGO = DIR_TRABAJO / "datos"
    assert (DIR_CODIGO / "comun.py").is_file(), f"No encontré comun.py en {DIR_CODIGO}"
sys.path.insert(0, str(DIR_CODIGO))
import comun

HUB = comun.Hub(REPO_HF, DIR_TRABAJO)   # REPO_HF = None: no sube ni baja nada
for relativa in (comun.RUTA_RAG, comun.RUTA_PARES, comun.RUTA_SPLIT):
    try:
        print("✓", comun.ruta_dato(RAIZ, relativa))
    except FileNotFoundError as e:
        print(f"✗ {e}")
print("Datos:", RAIZ, "| salidas:", DIR_TRABAJO, "| Hub:", REPO_HF or "desactivado")

In [ ]:
# Estilo de gráficos: una serie en azul, grilla y ejes discretos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

AZUL, NARANJA, TINTA, TINTA_2, GRILLA = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"axes.edgecolor": GRILLA, "axes.labelcolor": TINTA_2, "xtick.color": TINTA_2,
                     "ytick.color": TINTA_2, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRILLA, "grid.linewidth": 0.6, "axes.axisbelow": True,
                     "font.size": 10, "axes.titlesize": 11})
pd.set_option("display.max_colwidth", 200)


def barras(serie, titulo, xlabel, log=False, fmt="{:,.0f}"):
    """Una serie por categoría. En escala log se dibujan puntos: el largo de una barra en log engaña."""
    s = serie.sort_values()
    fig, ax = plt.subplots(figsize=(7, 0.4 * len(s) + 1.2))
    if log:
        ax.plot(s.values, s.index.astype(str), "o", color=AZUL, markersize=8)
        ax.set_xscale("log")
        ax.grid(axis="y", visible=True)
    else:
        ax.barh(s.index.astype(str), s.values, color=AZUL, height=0.6)
        ax.grid(axis="y", visible=False)
    for y, v in enumerate(s.values):
        ax.text(v, y, " " + fmt.format(v), va="center", color=TINTA_2, fontsize=9)
    ax.set_title(titulo, loc="left", color=TINTA)
    ax.set_xlabel(xlabel)
    plt.tight_layout()
    plt.show()


def curva_perdida(historial, titulo, ylabel, guardar=None):
    """Pérdida de entrenamiento (línea) y de validación (puntos) desde trainer.state.log_history."""
    hist = pd.DataFrame(historial)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    if "loss" in hist:
        tr = hist.dropna(subset=["loss"])
        ax.plot(tr.step, tr.loss, color=AZUL, linewidth=2, label="entrenamiento")
    if "eval_loss" in hist:
        ev = hist.dropna(subset=["eval_loss"])
        ax.plot(ev.step, ev.eval_loss, "o", color=NARANJA, markersize=8, markeredgecolor="white", label="validación")
    ax.set_xlabel("paso")
    ax.set_ylabel(ylabel)
    ax.set_title(titulo, loc="left", color=TINTA)
    ax.legend(frameon=False)
    plt.tight_layout()
    if guardar:
        fig.savefig(guardar, dpi=150)
    plt.show()

In [ ]:
assert CONDICION in ("solo_paso2", "paso1+2", "sin_ajuste", "solo_paso1", "it_solo_paso2", "it_sin_ajuste"), CONDICION
CHAT = CONDICION.startswith("it_")                 # instruct: plantilla de chat
MODELO = MODELO_IT if CHAT else MODELO_BASE
USA_PASO1 = CONDICION in ("paso1+2", "solo_paso1")
ENTRENA = CONDICION in ("paso1+2", "solo_paso2", "it_solo_paso2")
ADAPTADOR_PASO1 = DIR_TRABAJO / "paso1" / NOMBRE_PASO1 / "adaptador"
if USA_PASO1:
    HUB.bajar(ADAPTADOR_PASO1)
    assert (ADAPTADOR_PASO1 / "adapter_config.json").exists(), f"Falta el adaptador del paso 1 en {ADAPTADOR_PASO1}"

etiqueta = CONDICION if not ENTRENA else f"{CONDICION}__{'+'.join(MODULOS_PASO2)}_r{LORA_R}"
if USA_PASO1:
    etiqueta += f"__p1-{NOMBRE_PASO1}"
DIR_C = DIR_TRABAJO / "paso2" / etiqueta
DIR_C.mkdir(parents=True, exist_ok=True)
print("Condición:", etiqueta, "| modelo:", MODELO, "| plantilla de chat:", CHAT, "| salidas:", DIR_C)

## 1. Datos

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODELO)
pares = comun.cargar_pares(RAIZ)
por_split = {s: pares[pares.split == s].reset_index(drop=True) for s in ("train", "val", "test")}
display(pares.groupby("split").agg(pares=("clave", "size"), expresiones=("clave", "nunique")))

# Split oficial: las expresiones se comparten entre splits a propósito (4 pares en train, 1 en val o test);
# lo que no puede repetirse es un par ni una frase chilena
frases = {s: set(d.frase_chilena.str.strip().str.lower()) for s, d in por_split.items()}
assert not (frases["train"] & frases["val"]) and not (frases["train"] & frases["test"])     and not (frases["val"] & frases["test"]), "fuga entre splits: una frase está en dos splits"
display(por_split["train"][["termino", "frase_chilena", "frase_neutralizada"]].sample(5, random_state=SEED))

In [ ]:
from datasets import Dataset


def a_dataset(df):
    return Dataset.from_list([comun.tokenizar_par(c, n, tok, MAX_LEN, chat=CHAT)
                              for c, n in zip(df.frase_chilena, df.frase_neutralizada)])


ds_train, ds_val = a_dataset(por_split["train"]), a_dataset(por_split["val"])
largos = [len(x) for x in ds_train["input_ids"]]
print(f"train: {len(ds_train)} | val: {len(ds_val)} | tokens por ejemplo: mediana {np.median(largos):.0f}, máx {max(largos)}")
assert max(largos) < MAX_LEN, "algún ejemplo se truncó: sube MAX_LEN"

e = ds_train[0]
print("\n--- Entrada completa ---\n" + tok.decode(e["input_ids"]))
print("--- Parte con pérdida ---\n" + tok.decode([t for t in e["labels"] if t != -100]))

## 2. Modelo

In [ ]:
torch.manual_seed(SEED)
modelo = comun.cargar_modelo(MODELO, adaptador_a_fusionar=str(ADAPTADOR_PASO1) if USA_PASO1 else None, dtype=DTYPE)
if ENTRENA:
    modelo = comun.agregar_lora(modelo, MODULOS_PASO2, LORA_R, LORA_ALPHA, LORA_DROPOUT)

## 3. Entrenamiento

Evaluación al final de cada época y se queda el checkpoint con menor pérdida de validación. Las condiciones sin entrenamiento saltan esta sección.

In [ ]:
from transformers import Trainer, TrainingArguments

if ENTRENA:
    args = TrainingArguments(
        output_dir=str(DIR_C / "checkpoints"),
        per_device_train_batch_size=LOTE,
        per_device_eval_batch_size=LOTE,
        num_train_epochs=EPOCAS,
        max_steps=MAX_PASOS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_steps=0.05,
        weight_decay=0.0,
        bf16=DTYPE == "bfloat16",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        remove_unused_columns=False,
        report_to="none",
        seed=SEED,
    )
    trainer = Trainer(model=modelo, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                      data_collator=comun.ColadorPares(tok.pad_token_id), callbacks=[comun.callback_progreso()])
    trainer.train()
    trainer.save_model(str(DIR_C / "adaptador"))
    HUB.subir(DIR_C / "adaptador", reemplazar=True)

    curva_perdida(trainer.state.log_history, f"Paso 2 ({CONDICION}): pérdida", "pérdida sobre la respuesta",
                  guardar=DIR_C / "perdida.png")
else:
    print(f"{CONDICION}: sin entrenamiento, se evalúa el modelo tal cual")

## 4. Evaluación en validación y prueba

Generación greedy con el mismo prompt; se corta en `<eos>` o en el primer salto de línea. Métricas:
- **BLEU** y **chrF++** contra la referencia neutra (sacrebleu). Miden coincidencia de superficie, no significado: son secundarias frente al juez LLM y la evaluación humana del plan.
- **expresion_en_prediccion_%**: cuántas salidas aún contienen la expresión chilena literal (más bajo es mejor). Es coincidencia exacta de la forma base, así que no capta conjugaciones (*cogotearon*, *craneando*). Por eso léela junto a **expresion_en_chilena_%**, que es la misma medida sobre la entrada y sirve de techo.
- **copia_literal_%**: cuántas salidas son idénticas a la entrada.

Usa **validación** para comparar configuraciones y deja **prueba** para el resultado final.

In [ ]:
resultados = {}
for split in ("val", "test"):
    df = por_split[split].copy()
    df["prediccion"] = comun.traducir(modelo, tok, df.frase_chilena.tolist(), bs=32, chat=CHAT)
    df[["id_mistral", "clave", "termino", "frase_chilena", "frase_neutralizada", "prediccion"]].to_csv(
        DIR_C / f"predicciones_{split}.csv", index=False)
    resultados[split] = comun.metricas_traduccion(df)
    por_split[split] = df

with open(DIR_C / "metricas.json", "w", encoding="utf-8") as f:
    json.dump({"condicion": CONDICION, "etiqueta": etiqueta, "metricas": resultados,
               "config": {"modelo": MODELO, "chat": CHAT, "paso1": NOMBRE_PASO1 if USA_PASO1 else None,
                          "modulos": MODULOS_PASO2 if ENTRENA else None, "r": LORA_R, "alpha": LORA_ALPHA,
                          "lr": LR, "epocas": EPOCAS, "lote": LOTE, "seed": SEED},
               "historial": trainer.state.log_history if ENTRENA else None}, f, indent=2, ensure_ascii=False)
HUB.subir(DIR_C, ignorar=["checkpoints/*", "adaptador/*"], mensaje=f"Resultados de {etiqueta}")
display(pd.DataFrame(resultados).T)

In [ ]:
# Ejemplos de prueba
display(por_split["test"].sample(12, random_state=SEED)[["termino", "frase_chilena", "frase_neutralizada", "prediccion"]])

## 5. Comparación entre condiciones

Lee todas las condiciones que ya se corrieron, también las de otras sesiones (cada una dejó `metricas.json` en `paso2/` del repo del Hub). Ejecuta esta sección al final, después de correr al menos `solo_paso2`, `paso1+2` e `it_solo_paso2`.

In [ ]:
HUB.bajar("paso2/*/metricas.json", "paso2/*/predicciones_test.csv")   # condiciones corridas en otras sesiones
filas = []
for ruta in sorted((DIR_TRABAJO / "paso2").glob("*/metricas.json")):
    m = json.loads(ruta.read_text(encoding="utf-8"))
    for split, met in m["metricas"].items():
        filas.append({"etiqueta": m["etiqueta"], "condicion": m["condicion"], "split": split, **met})
comparacion = pd.DataFrame(filas)
display(comparacion.pivot_table(index="etiqueta", columns="split",
                                values=["chrf++", "bleu", "expresion_en_prediccion_%", "copia_literal_%"]).round(1))

In [ ]:
# Pares de prueba donde las condiciones entrenadas difieren (para revisión humana o del juez)
PRINCIPALES = ("solo_paso2", "paso1+2", "it_solo_paso2")
tabla = None
for cond in PRINCIPALES:
    rutas = sorted((DIR_TRABAJO / "paso2").glob(f"{cond}__*/predicciones_test.csv"))
    if not rutas:
        print("Falta correr:", cond)
        continue
    p = pd.read_csv(rutas[-1]).rename(columns={"prediccion": cond})
    tabla = p if tabla is None else tabla.merge(p[["id_mistral", cond]], on="id_mistral")
if tabla is not None:
    conds = [c for c in PRINCIPALES if c in tabla]
    distintos = tabla[tabla[conds].apply(lambda r: r.str.strip().nunique() > 1, axis=1)]
    print(f"Las condiciones {conds} difieren en {len(distintos)} de {len(tabla)} pares de prueba")
    display(distintos.sample(min(15, len(distintos)), random_state=SEED)[
        ["termino", "frase_chilena", "frase_neutralizada", *conds]])